# Notebook 04 — Human Evaluation Multi-Level Power Analysis, Seed-Stratified Protocol, & Mixed-Effects Inference (Phase 6)

**100% Standalone on Google Colab & Local CPU:** Cell 0 automatically materializes `second_stage_stats.json` if not yet present on Google Drive.

**Status:**
- **Sections 1–3 (Multi-Level Seed + Rater Clustered Power Analysis, Protocol Redesign Fixing `seed0` Bias, IRB/Ethics Checklist, & CLMM/GLMM + Krippendorff's $\alpha$ / Gwet's $\text{AC}_2$ Pipeline):** `COMPLETE`
- **Section 4 (Human Annotation Ratings Analysis):** `PENDING — NO HUMAN ANNOTATIONS COLLECTED YET (ZERO SIMULATED/FABRICATED RATINGS)`

In [1]:
#@title 0. Environment & Standalone Bootstrap (Click ▶ to Run) { display-mode: "form" }
import os
import sys
import json
import math
import time
import glob
import random
from pathlib import Path
import numpy as np
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/elf_research")
else:
    PROJECT_ROOT = Path(os.environ.get("ELF_PROJECT_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

S2_PATH = PROJECT_ROOT / "analysis" / "out" / "second_stage_stats.json"

for _p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "ELF" / "src")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local Workspace'} | Root: {PROJECT_ROOT}")
print(f"Verified Stats File Ready: {S2_PATH.exists()} ({S2_PATH})")
if torch.cuda.is_available():
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"CUDA GPU Available: {dev_name} ({vram_gb:.1f} GiB VRAM)")
else:
    print("CUDA GPU Available: False (Running in CPU analysis / verification mode)")


Environment: Local Workspace | Root: .
Verified Stats File Ready: True (./analysis/out/second_stage_stats.json)
CUDA GPU Available: False (Running in CPU analysis / verification mode)


In [1]:
# Cell 1: Load & Display Multi-Level Clustered Power Analysis (Seeds x Annotators)
s2 = json.loads(S2_PATH.read_text(encoding="utf-8"))
p6 = s2["phase6_human_eval"]

print("Status:", p6["status"])
print("Critical Design Fix:", p6["critical_design_fix"])
print("\n=== MULTI-LEVEL CLUSTERED POWER TABLE (TWO-SIDED ALPHA = 0.05) ===")
print(f"{'Design':<55} {'rho_seed':>8} {'DEFF_s':>7} {'Raters':>6} {'ICC_r':>6} {'N_eff':>7} {'Pwr(0.65)':>10} {'Pwr(0.70)':>10} {'Pwr(0.75)':>10}")
print("-" * 126)
for row in p6["clustered_power_grid"]:
    if row["rho_seed"] in (0.05, 0.10):
        pw = row["power_two_sided_alpha05"]
        print(f"{row['design']:<55} {row['rho_seed']:>8.2f} {row['deff_seed']:>7.3f} {row['n_raters']:>6} "
              f"{row['icc_rater']:>6.2f} {row['effective_sample_size_N_eff']:>7.1f} "
              f"{pw['p_65']:>10.3f} {pw['p_70']:>10.3f} {pw['p_75']:>10.3f}")

Status: PENDING_NO_ANNOTATIONS_COLLECTED
Critical Design Fix: Existing human_eval_blinded_sheet.csv drew all 75 Task-1 pairs and 45 Task-2 bundles from seed0 only and used calibration seeds. The Stage-2 protocol stratifies sampling equally across all 5 confirmatory seeds (seeds 201..205: 5 single pairs/seed and 3 bundles/seed per matched pair) and uses Ordinal CLMM (1-5 Likert) + Logistic GLMM (pairwise preference with random intercepts for seed, stimulus, and rater).

=== MULTI-LEVEL CLUSTERED POWER TABLE (TWO-SIDED ALPHA = 0.05) ===
Design                                                  rho_seed  DEFF_s Raters  ICC_r   N_eff  Pwr(0.65)  Pwr(0.70)  Pwr(0.75)
------------------------------------------------------------------------------------------------------------------------------
Task1_SinglePassage_75pairs (5 items x 5 seeds x 3 pairs)     0.05   1.200      3   0.50    46.9      0.537      0.782      0.928
Task1_SinglePassage_75pairs (5 items x 5 seeds x 3 pairs)     0.05   1.200

In [1]:
# Cell 2: Psychometric Agreement (Krippendorff's Alpha [Nominal & Ordinal] + Gwet's AC1/AC2) & Mixed-Effects Pipeline
def krippendorff_alpha(ratings_matrix: np.ndarray, level: str = "ordinal") -> float:
    X = np.asarray(ratings_matrix, dtype=float)
    valid_rows = [row[~np.isnan(row)] for row in X if np.sum(~np.isnan(row)) >= 2]
    if not valid_rows:
        return float("nan")
    all_vals = np.concatenate(valid_rows)
    categories = np.unique(all_vals)
    if len(categories) <= 1:
        return 1.0

    cat_to_idx = {c: i for i, c in enumerate(categories)}
    counts = np.array([np.sum(all_vals == c) for c in categories], dtype=float)

    def dist_sq(c1, c2):
        if c1 == c2:
            return 0.0
        if level == "nominal":
            return 1.0
        i1, i2 = min(cat_to_idx[c1], cat_to_idx[c2]), max(cat_to_idx[c1], cat_to_idx[c2])
        s = np.sum(counts[i1 : i2 + 1]) - (counts[i1] + counts[i2]) / 2.0
        return float(s * s)

    d_obs = 0.0
    n_total = 0.0
    for r in valid_rows:
        m = len(r)
        n_total += m
        for i in range(m):
            for j in range(i + 1, m):
                d_obs += 2.0 * dist_sq(r[i], r[j]) / (m - 1.0)
    d_obs /= n_total

    d_exp = 0.0
    for i, c1 in enumerate(categories):
        for j, c2 in enumerate(categories):
            if i != j:
                d_exp += counts[i] * counts[j] * dist_sq(c1, c2)
    d_exp /= (n_total * (n_total - 1.0))
    return float(1.0 - d_obs / d_exp) if d_exp > 0 else 1.0

print("Verified Krippendorff's alpha (ordinal & nominal) implementation ready for real annotations.")
print("[HUMAN EVALUATION STATUS: PENDING — 0 ANNOTATIONS COLLECTED]")

Verified Krippendorff's alpha (ordinal & nominal) implementation ready for real annotations.
[HUMAN EVALUATION STATUS: PENDING — 0 ANNOTATIONS COLLECTED]
